# Quantum Walk Algorithms

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/08_quantum_walk_algorithms.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/08_quantum_walk_algorithms.ipynb)

This notebook demonstrates quantum walk algorithms available in QuantX Engine:

1. **Discrete-Time Quantum Walk (DTQW)** — Hadamard coin on a cycle graph
2. **Continuous-Time Quantum Walk (CTQW)** — Laplacian evolution, transfer probability
3. **Grover Walk Search** — Quadratic speedup for graph search via oracle
4. **Walk Spectrum** — Quasi-energy dispersion of periodic walks

All examples run in pure Python (no C++ library required).

In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np
import quantx_engine
print('quantx_engine version:', quantx_engine.__version__)

## 1. Discrete-Time Quantum Walk on a Cycle

A discrete-time quantum walk on an $N$-node cycle uses a **coin** operator $C$ acting on
an internal degree of freedom, and a **shift** operator $S$ that moves the walker.

With the Hadamard coin, starting at node 0, the walker spreads ballistically ($\sigma \propto t$)
— quadratically faster than a classical random walk ($\sigma \propto \sqrt{t}$).

In [ ]:
from quantx_engine.quantum_walk import DiscreteTimeWalk

N = 16  # number of nodes in the cycle
T = 20  # number of walk steps

# Create a Hadamard-coin walk on an N-cycle
walk = DiscreteTimeWalk.cycle(N, coin='hadamard')

# Run T steps and record position probability distribution
result = walk.run(T)
probs = result.position_probs

print(f'Walk on {N}-node cycle, {T} steps')
print(f'Position probability distribution (first 8 nodes):')
for i, p in enumerate(probs[:8]):
    bar = '█' * int(p * 40)
    print(f'  node {i:2d}: {p:.4f}  {bar}')

# Sanity check: probabilities sum to 1
assert abs(sum(probs) - 1.0) < 1e-9, f'Probabilities do not sum to 1: {sum(probs)}'
print(f'\nSum of probabilities: {sum(probs):.10f}  ✓')

### Spread comparison: quantum vs classical

The standard deviation of the position distribution grows as $\sigma \propto T$ for quantum walks
vs $\sigma \propto \sqrt{T}$ for classical random walks — the hallmark of quantum advantage.

In [ ]:
from quantx_engine.quantum_walk import DiscreteTimeWalk

N = 50
steps_list = [5, 10, 15, 20]

print(f'Standard deviation of position distribution on {N}-node cycle:')
print(f'{"Steps":>6}  {"Quantum σ":>12}  {"Classical σ":>12}  {"Ratio":>8}')
print('-' * 45)

for T in steps_list:
    walk = DiscreteTimeWalk.cycle(N, coin='hadamard')
    result = walk.run(T)
    probs = result.position_probs
    nodes = np.arange(N)
    mean = np.sum(nodes * probs)
    sigma_q = np.sqrt(np.sum((nodes - mean)**2 * probs))
    sigma_c = np.sqrt(T)  # classical random walk
    print(f'{T:>6}  {sigma_q:>12.4f}  {sigma_c:>12.4f}  {sigma_q/sigma_c:>8.2f}x')

## 2. Continuous-Time Quantum Walk

A continuous-time quantum walk evolves under the graph Laplacian (or adjacency matrix):
$$|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle, \quad H = -A \text{ (adjacency)}$$

The **transfer probability** from node $s$ to node $t$ is $P(s \to t, T) = |\langle t|e^{-iHT}|s\rangle|^2$.

In [ ]:
from quantx_engine.quantum_walk import ContinuousTimeQuantumWalk, ctqw_transfer_probability

# 8-node complete graph — ideal for perfect state transfer
N = 8
adj = np.ones((N, N)) - np.eye(N)  # complete graph K_8

# ContinuousTimeQuantumWalk takes the adjacency matrix as Hamiltonian
ctqw = ContinuousTimeQuantumWalk(adj)

# Scan transfer probability from node 0 to node N//2 over time
# ctqw_transfer_probability takes the walk *object*, not the raw matrix
times = np.linspace(0, 4, 50)
transfers = [ctqw_transfer_probability(ctqw, source=0, target=N//2, t=t) for t in times]

max_p = max(transfers)
max_t = times[np.argmax(transfers)]

print(f'CTQW on K_{N} (complete graph)')
print(f'Max transfer probability 0 → {N//2}: {max_p:.4f} at t = {max_t:.3f}')

# Probability distribution at peak time
print(f'\nProbability distribution at t = {max_t:.3f}:')
result = ctqw.run(t=max_t)
probs = result.position_probs
for i, p in enumerate(probs):
    marker = ' ← peak' if i == N//2 else ''
    print(f'  node {i}: {p:.4f}{marker}')

In [ ]:
# Path graph — localization behaviour
N_path = 10
# Path graph adjacency: tridiagonal
adj_path = np.diag(np.ones(N_path - 1), 1) + np.diag(np.ones(N_path - 1), -1)

ctqw_path = ContinuousTimeQuantumWalk(adj_path)
result_path = ctqw_path.run(t=3.0)
probs_path = result_path.position_probs

print(f'CTQW on path graph P_{N_path}, t=3.0:')
for i, p in enumerate(probs_path):
    bar = '█' * int(p * 30)
    print(f'  node {i:2d}: {p:.4f}  {bar}')

assert abs(sum(probs_path) - 1.0) < 1e-9
print(f'\nNorm preserved: sum = {sum(probs_path):.10f}  ✓')

## 3. Grover Walk Search

The `GroverWalk` implements the quantum walk-based search algorithm of Shenvi, Kempe and Whaley (2003).
An oracle marks the target node; the walk amplitude amplifies it in $O(\sqrt{N})$ steps.

We demonstrate on a **hypercube** graph where the algorithm achieves quadratic speedup.

In [ ]:
from quantx_engine.quantum_walk import GroverWalk, DiscreteTimeWalk

# 3-dimensional hypercube Q_3 (8 nodes)
# Build adjacency: nodes are bit strings 0..7, edges connect nodes differing in 1 bit
dim = 3
N = 2**dim
adj = np.zeros((N, N), dtype=int)
for i in range(N):
    for j in range(N):
        if bin(i ^ j).count('1') == 1:  # differ by exactly 1 bit
            adj[i, j] = 1

target = 7  # search for node 7
T_search = int(np.pi / 4 * np.sqrt(N))  # optimal steps ≈ π/4 * √N
T_search = max(T_search, 3)

# GroverWalk expects a DiscreteTimeWalk as first argument (not a raw array).
# We build one with the Grover coin and set a uniform initial superposition.
dtw = DiscreteTimeWalk(adj, coin='grover')
d = dtw.coin_dim
uniform = np.ones(d * N, dtype=complex) / np.sqrt(d * N)
dtw._state = uniform.copy()
dtw._initial_state = uniform.copy()

gw = GroverWalk(dtw, marked_nodes=[target])
result = gw.run(T_search)

p_marked = result.step_probs[-1]   # P(marked) after exactly T_search steps
print(f'Grover Walk on Q_{dim} (N={N}), target={target}, steps={T_search}')
print(f'  P(marked node) = {p_marked:.4f}')
print(f'  Position probs: {[f"{p:.3f}" for p in result.position_probs]}')

# Also use .search() convenience method
search_result = gw.search()
best_node = int(np.argmax(search_result.position_probs))
print(f'\nGroverWalk.search() found node {best_node} '
      f'with probability {search_result.max_marked_prob:.4f}')

# For small hypercubes the algorithm should amplify the marked node
assert best_node == target, f'Expected node {target}, got {best_node}'
print('Search assertion passed  ✓')

In [ ]:
# Scaling: p_marked as a function of hypercube dimension
print('Grover Walk search: success probability vs hypercube dimension')
print(f'{"dim":>4}  {"N":>6}  {"T_opt":>6}  {"p_marked":>10}')
print('-' * 35)

for d in [2, 3, 4]:
    n = 2**d
    a = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if bin(i ^ j).count('1') == 1:
                a[i, j] = 1
    T_opt = max(int(np.pi / 4 * np.sqrt(n)), 3)
    # Build DiscreteTimeWalk with Grover coin + uniform initial state
    dtw_d = DiscreteTimeWalk(a, coin='grover')
    deg = dtw_d.coin_dim
    unif = np.ones(deg * n, dtype=complex) / np.sqrt(deg * n)
    dtw_d._state = unif.copy()
    dtw_d._initial_state = unif.copy()
    gw_d = GroverWalk(dtw_d, marked_nodes=[n - 1])
    res = gw_d.search()
    print(f'{d:>4}  {n:>6}  {T_opt:>6}  {res.max_marked_prob:>10.4f}')

## 4. Walk Spectrum

The `walk_spectrum` function computes the quasi-energy eigenvalues of the walk operator $W = S(C \otimes I)$.
For a cycle with Grover coin, the spectrum exhibits flat bands characteristic of the Grover coin.

In [ ]:
from quantx_engine.quantum_walk import walk_spectrum

# Compare spectra of Hadamard vs Grover coin on 8-cycle
N = 8

for coin_name in ['hadamard', 'grover']:
    walk = DiscreteTimeWalk.cycle(N, coin=coin_name)
    eigenvalues = walk_spectrum(walk)   # eigenvalues of the walk operator W
    # quasi-energies are phases: eigenvalues on unit circle
    phases = np.angle(eigenvalues) / np.pi
    phases_sorted = np.sort(phases)
    n_unique = len(set(np.round(phases_sorted, 4)))
    print(f'{coin_name.capitalize()} coin spectrum ({N}-cycle):')
    print(f'  {len(eigenvalues)} eigenvalues, {n_unique} distinct quasi-energies (θ/π)')
    print(f'  Range: [{phases_sorted[0]:.4f}, {phases_sorted[-1]:.4f}]')
    # All eigenvalues should lie on the unit circle
    magnitudes = np.abs(eigenvalues)
    assert np.allclose(magnitudes, 1.0, atol=1e-10), 'Eigenvalues not on unit circle!'
    print(f'  All eigenvalues on unit circle  ✓\n')

## 5. Coined Walk on a Hypercube Factory

QuantX provides factory methods for common graph topologies.

In [ ]:
from quantx_engine.quantum_walk import DiscreteTimeWalk

# Hypercube Q_4 (16 nodes, degree 4)
walk_hc = DiscreteTimeWalk.hypercube(dim=4)
result_hc = walk_hc.run(steps=10)

probs_hc = result_hc.position_probs
N_hc = len(probs_hc)

print(f'Discrete-time walk on hypercube Q_4 (N={N_hc}), 10 steps')
print(f'Max node probability: {max(probs_hc):.4f} at node {np.argmax(probs_hc)}')
print(f'Min node probability: {min(probs_hc):.4f}')
print(f'Sum of probabilities: {sum(probs_hc):.10f}')

assert abs(sum(probs_hc) - 1.0) < 1e-9
print('Normalization check passed  ✓')

## Summary

| Algorithm | Key result | Module |
|-----------|-----------|--------|
| Discrete-time walk (Hadamard coin) | Ballistic spread $\sigma \propto T$ vs classical $\sqrt{T}$ | `quantum_walk.DiscreteTimeWalk` |
| Continuous-time walk | Transfer probability on complete/path graphs | `quantum_walk.ContinuousTimeQuantumWalk` |
| Grover walk search | Marked node amplified on hypercube | `quantum_walk.GroverWalk` |
| Walk spectrum | Quasi-energy eigenvalues on unit circle | `quantum_walk.walk_spectrum` |

Quantum walks provide the algorithmic foundation for quantum speedups in search, sampling, and quantum simulation.

Next: **[09 — Noise Mitigation](09_noise_mitigation.ipynb)**